In [21]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import json

from src.utils.utils import find_project_root, load_ligand_models

BASE_DIR = find_project_root()

DATA_DIR = BASE_DIR / "data" / "processed" 

from figures.fig_scripts.fig3_functions import *

### Load BMP4 model:

In [4]:
models = load_ligand_models(subset=["BMP4"])
bmp4_model = models["BMP4"]

del models

### Load gene to cluster mappings from figure 2:

In [ ]:
#Generated in figure_2.ipynb
cluster_dict = json.load(open(DATA_DIR / "dicts" / "kmeans_5_clusters_dict.json", "r"))

### Generate ligand gene list (up & down not separated) and run GO:

In [7]:
background_genes = bmp4_model.df.columns.to_list()

### Run GO numbered clusters - UPDATED clustering

In [ ]:
#print length of each cluster's gene list
for cluster_num, genes in cluster_dict.items():
    print(f"Cluster {cluster_num}: {len(genes)} genes")

Cluster 1: 113 genes
Cluster 2: 341 genes
Cluster 3: 64 genes
Cluster 4: 243 genes
Cluster 5: 16 genes


In [ ]:
go_results_num_clusters_dict = run_go_analysis(
    cluster_dict, background_genes, seperate_up_down=False
)

Processing 1...
Processing 2...
Processing 3...
Processing 4...
Processing 5...


In [10]:
save_go_results_dataframe(
    go_results_num_clusters_dict,
    DATA_DIR / "go_analysis" / "go_dataframes" ,
)

In [11]:
go_results_num_clusters_dict = load_go_dataframes_dict(
    path=DATA_DIR / "go_analysis" / "go_dataframes" 
)

In [17]:
go_results_num_clusters_json = generate_json_for_R(go_results_num_clusters_dict)
save_json_for_R(
    go_results_num_clusters_json,
    DATA_DIR / "go_analysis" / "go_jsons" / "kmeans_5_clusters_go_terms.json",
)

In [18]:
env_path = "/home/labs/antebilab/guyilan/.conda/envs/paper_env"

os.environ['PATH'] = f"{env_path}/bin:" + os.environ['PATH']

# Define your custom paths here
my_input = DATA_DIR / "go_analysis" / "go_jsons" / "kmeans_5_clusters_go_terms.json"
my_output = DATA_DIR / "go_analysis" / "go_jsons" / "sim_results_kmeans_5_clusters_from_R.json"
r_script = BASE_DIR / "src" / "analysis" / "get_similarities.R"

# Execute R with the paths as arguments
!Rscript {r_script} {my_input} {my_output}


GOSemSim v2.36.0 Learn more at https://yulab-smu.top/contribution-knowledge-mining/

Please cite:

Guangchuang Yu, Fei Li, Yide Qin, Xiaochen Bo, Yibo Wu and Shengqi
Wang. GOSemSim: an R package for measuring semantic similarity among GO
terms and gene products. Bioinformatics. 2010, 26(7):976-978

Attaching package: ‘igraph’

The following objects are masked from ‘package:stats’:

    decompose, spectrum

The following object is masked from ‘package:base’:

    union

Loading required package: AnnotationDbi
Loading required package: stats4
Loading required package: BiocGenerics
Loading required package: generics

Attaching package: ‘generics’

The following objects are masked from ‘package:igraph’:

    components, union

The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union


Attaching package: ‘BiocGenerics’

The following objects are masked from ‘package:igraph’:

    normalize, path

The 

numbered_clusters_go_terms.json is processed in src/analysis/get_similarities.R and returend as sim_results_num_clusters_from_R.json

In [19]:
with open(
    DATA_DIR / "go_analysis" / "go_jsons" / "sim_results_kmeans_5_clusters_from_R.json", "r"
) as f:
    go_sim_num_clusters_data = json.load(f)